# Per-step loss timing tests

This notebook measures 2D Allen--Cahn optimization cost after compilation. `rom = true` constructs a ROM before measurement; the reported times exclude that construction.


In [1]:
repo_root = normpath(joinpath(@__DIR__, "..", ".."))
import Pkg
Pkg.activate(joinpath(repo_root, "Julia"))
include(joinpath(repo_root, "src", "Tools", "Tests", "timing_tests.jl"))


  Activating 

[package-load] Loading LinearAlgebra
[package-load] Loaded LinearAlgebra
[package-load] Loading SparseArrays
[package-load

project at `~/Library/CloudStorage/OneDrive-RiceUniversity/Documents/VSCode_Projects/Brookhaven_main/Julia`


] Loaded SparseArrays
[package-load] Loading Statistics
[package-load] Loaded Statistics
[package-load] Loading Random
[package-load] Loaded Random
[package-load] Loading ComponentArrays
[package-load] Loaded ComponentArrays
[package-load] Loading LinearSolve
[package-load] Loaded LinearSolve
[package-load] Loading OrdinaryDiffEq
[package-load] Loaded OrdinaryDiffEq
[package-load] Loading OrdinaryDiffEqSDIRK
[package-load] Loaded OrdinaryDiffEqSDIRK
[package-load] Loading OrdinaryDiffEqLowOrderRK
[package-load] Loaded OrdinaryDiffEqLowOrderRK
[package-load] Loading SciMLSensitivity
[package-load] Loaded SciMLSensitivity
[package-load] Loading ADTypes
[package-load] Loaded ADTypes
[package-load] Loading Zygote
[package-load] Loaded Zygote
[package-load] Loading Mooncake
[package-load] Loaded Mooncake
[package-load] Loading Optimization
[package-load] Loaded Optimization
[package-load] Loading OptimizationOptimisers
[package-load] Loaded OptimizationOptimisers
[package-load] Loading Opti

print_timing_table

## 1. Make a schedule

The required first argument is a test name. Results are saved under `Untracked/Tests/timing_test_results/<test name>/`: `cases.txt`, one text and one `.jls` file per metric, and combined `results.txt`/`results.jls` after a complete suite. Pass arrays to create a Cartesian product. Omitted keywords use one default value. `loss_space = ["REDUCED"]` requires `rom = [true]`. `tfinal` defaults to each `window_T`, so reference/ROM construction is outside the timed step.


In [4]:
schedule = timing_schedule("ac_rom_scaling_with_n_and_window_T_full_simplified_loss";
    N=[32,64,128,256],
    h=[8],
    window_N_obs=[10],
    window_T=[10.0],
    r=[10],
    m=[10],
    rom=[true],
    loss_space=["FULL"],
)


[2026-07-22 02:53:33] Timing schedule with 4 combination(s):
[2026-07-22 02:53:33] test_name | N | h | window_N_obs | window_T | r | m | rom | loss_space | tfinal | seed
[2026-07-22 02:53:33] ac_rom_scaling_with_n_and_window_T_full_simplified_loss | 32 | 8 | 10 | 10.0 | 10 | 10 | true | FULL | 10.0 | 1
[2026-07-22 02:53:33] ac_rom_scaling_with_n_and_window_T_full_simplified_loss | 64 | 8 | 10 | 10.0 | 10 | 10 | true | FULL | 10.0 | 1
[2026-07-22 02:53:33] ac_rom_scaling_with_n_and_window_T_full_simplified_loss | 128 | 8 | 10 | 10.0 | 10 | 10 | true | FULL | 10.0 | 1
[2026-07-22 02:53:33] ac_rom_scaling_with_n_and_window_T_full_simplified_loss | 256 | 8 | 10 | 10.0 | 10 | 10 | true | FULL | 10.0 | 1
[2026-07-22 02:53:33] Saved 4 timing case(s) to /Users/milesgantcher/Library/CloudStorage/OneDrive-RiceUniversity/Documents/VSCode_Projects/Brookhaven_main/Untracked/Tests/timing_test_results/ac_rom_scaling_with_n_and_window_T_full_simplified_loss/cases.txt.


4-element Vector{NamedTuple}:
 (test_name = "ac_rom_scaling_with_n_and_window_T_full_simplified_loss", N = 32, h = 8, window_N_obs = 10, window_T = 10.0, r = 10, m = 10, rom = true, loss_space = "FULL", tfinal = 10.0, seed = 1)
 (test_name = "ac_rom_scaling_with_n_and_window_T_full_simplified_loss", N = 64, h = 8, window_N_obs = 10, window_T = 10.0, r = 10, m = 10, rom = true, loss_space = "FULL", tfinal = 10.0, seed = 1)
 (test_name = "ac_rom_scaling_with_n_and_window_T_full_simplified_loss", N = 128, h = 8, window_N_obs = 10, window_T = 10.0, r = 10, m = 10, rom = true, loss_space = "FULL", tfinal = 10.0, seed = 1)
 (test_name = "ac_rom_scaling_with_n_and_window_T_full_simplified_loss", N = 256, h = 8, window_N_obs = 10, window_T = 10.0, r = 10, m = 10, rom = true, loss_space = "FULL", tfinal = 10.0, seed = 1)

## 2. Run repeated timings

Set `RUN_BENCHMARKS = true` when ready. Each workload is run once to compile, then timed repeatedly with `@timed`; results include mean, standard deviation, range, allocated bytes, and raw samples. `forward` measures the ODE solve, `loss` measures comparison/reconstruction from a precomputed solution, and `adam_step` measures forward + reverse + one Adam update.


In [5]:
RUN_BENCHMARKS = true
repeats = 5

if RUN_BENCHMARKS
    timing_results = time_optimization_steps(schedule; repeats)
    print_timing_table(timing_results.forward)
    print_timing_table(timing_results.loss)
    print_timing_table(timing_results.adam_step)
end


[2026-07-22 02:53:37] Starting forward, loss, and Adam-step timing suite.
[2026-07-22 02:53:37] Preparing 4 timing case(s); reference and ROM construction are excluded from measurements.
[2026-07-22 02:53:37] Preparing N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL: building reference trajectory.
[2026-07-22 02:53:47] Preparing N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL: building ROM training problem.
[2026-07-22 02:53:50] Preparing N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL: complete; timed work has not started.
[2026-07-22 02:53:51] Preparing N=64, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL: building reference trajectory.
[2026-07-22 02:53:51] Preparing N=64, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL: building ROM training problem.
[2026-07-22 02:53:52] Preparing N=64, h=8, window_N_obs=10, window_T=10.0, r=10, 

┌ Warning: tangent_to_primal!! is deprecated and will be removed in 0.6. Results may be inconsistent with the `friendly_tangents` opt-in mechanism (`FriendlyTangentCache`): types that override `friendly_tangent_cache` will not have their custom conversion applied.
└ @ Mooncake /Users/milesgantcher/.julia/packages/Mooncake/2qaRx/src/tangents/tangents.jl:1770


[2026-07-22 02:55:57] Adam step (N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL): warm-up complete; collecting 5 timed sample(s).
[2026-07-22 02:55:57] Adam step (N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL): timed sample 1/5 started.
[2026-07-22 02:55:58] Adam step (N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL): timed sample 1/5 complete in 0.02111 s, 26792496 bytes.
[2026-07-22 02:55:58] Adam step (N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL): timed sample 2/5 started.
[2026-07-22 02:55:58] Adam step (N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL): timed sample 2/5 complete in 0.02083 s, 26792496 bytes.
[2026-07-22 02:55:58] Adam step (N=32, h=8, window_N_obs=10, window_T=10.0, r=10, m=10, rom=true, loss_space=FULL): timed sample 3/5 started.
[2026-07-22 02:55:58] Adam step (N=32, h=8, window_N_obs=10, windo

4-element Vector{NamedTuple}:
 (test_name = "ac_rom_scaling_with_n_and_window_T_full_simplified_loss", N = 32, h = 8, window_N_obs = 10, window_T = 10.0, r = 10, m = 10, rom = true, loss_space = "FULL", tfinal = 10.0, seed = 1, metric = "adam_step", mean_seconds = 0.0208853332, std_seconds = 0.00013121995534483272, min_seconds = 0.020759791, max_seconds = 0.021105375, mean_bytes = 2.6792496e7, samples_seconds = [0.021105375, 0.02083225, 0.020887208, 0.020759791, 0.020842042], samples_bytes = [2.6792496e7, 2.6792496e7, 2.6792496e7, 2.6792496e7, 2.6792496e7])
 (test_name = "ac_rom_scaling_with_n_and_window_T_full_simplified_loss", N = 64, h = 8, window_N_obs = 10, window_T = 10.0, r = 10, m = 10, rom = true, loss_space = "FULL", tfinal = 10.0, seed = 1, metric = "adam_step", mean_seconds = 0.018032750000000004, std_seconds = 3.965884761941511e-5, min_seconds = 0.017969833, max_seconds = 0.0180715, mean_bytes = 3.3712912e7, samples_seconds = [0.017969833, 0.01802675, 0.018060875, 0.018071

`adam_step.mean_seconds - forward.mean_seconds - loss.mean_seconds` is a rough estimate of reverse-mode and update overhead. It is only approximate because full/reduced loss reconstruction and AD bookkeeping are not perfectly additive.
